# 19 - ¿Qué parte de la ventaja de nnU-Net se puede portar?

## La pregunta

La tesis (línea 911) enumera **cuatro** diferencias con nnU-Net y **no separa ninguna**:
resolución, intensidad de augmentation, duración del entrenamiento y normalización.
Este notebook mide dos de ellas y decide si una tercera está realmente descartada.

Lo que nnU-Net eligió para estos datos, leído de
`nnunet_workspace/nnUNet_preprocessed/Dataset501_VFSS/nnUNetPlans.json`:

```
patch_size            [1024, 1024]
normalization_schemes ["ZScoreNormalization"]
batch_dice            true
intensidades: mean 87.8  std 22.0  p0.5 37  p99.5 154
```

Y sus augmentations, leídas de `nnunetv2` instalado, contra las tuyas:

| | tuyo | nnU-Net 2D |
|---|---|---|
| rotación | **±5°** | **±180°**, p=0.2 |
| escala | **±3 %** | **0.7–1.4**, p=0.2 |
| espejo | apagado | activo |
| gamma | (0.9–1.1) p=0.2 | (0.7–1.5) p=0.1 y p=0.3 |
| desenfoque | no tiene | sigma 0.5–1, p=0.2 |
| baja resolución | no tiene | escala 0.5–1, p=0.25 |

## Los cuatro brazos

Un solo cambio por brazo sobre el supervisado. Nada se combina.

| brazo | cambio | por qué |
|---|---|---|
| **A** | ninguno (`baseline`) | el baseline **de esta sesión**, no el de julio |
| **D** | `batch_size = 1` | **el control de la resolución**, ver abajo |
| **B** | `image_norm = "zscore"` | la normalización de nnU-Net |
| **C1** | `aug_profile = "nnunet_moderate"` | más augmentation, geometría plausible |
| **C2** | `aug_profile = "nnunet_full"` | nnU-Net entero, incluido espejo y ±180° |

## Por qué el brazo D es el más importante de todos

Los runs a 1024 que hay en disco **no comparan resolución**, porque cambiaron dos cosas:

```
runs/supervised_1024*   [1024,1024] bs=1  F1 .7501 / .6964 / .8091 / .7429
runs_final_v1/supervised [320,320]  bs=5  F1 .8070 / .8158 / .7786
```

El modelo tiene **100 capas `BatchNorm2d`**; con `batch_size=1` cada una estima su
media y su varianza sobre una sola imagen. La dispersión lo confirma: std `.0566`
contra `.0194`, y **una semilla a 1024 dio `.8091`**, que iguala al baseline.

Entrenar a 320 con `bs=1` cuesta tres runs y decide la cuestión:
si cae a ~.75, el batch lo explica todo y **la resolución nunca se probó**.

## Por qué hay un brazo baseline en vez de usar `runs_final_v1`

Colab cambió de stack entre julio y septiembre de 2026:

```
runs_final_v1:  python 3.12.13  torch 2.2.1   albumentations 1.3.1  smp 0.3.3
hoy:            python 3.13.15  torch 2.11.0  albumentations 2.0.8  smp 0.5.0
```

Y la 2.x **descarta en silencio** argumentos que la 1.x aceptaba (`var_limit` de
`GaussNoise`, `value`/`mask_value` de `ShiftScaleRotate`). Comparar contra julio
mezclaría tu cambio con una actualización de librerías. Con el baseline entrenado
aquí, el entorno es común a los dos lados y se cancela.

## C1 y C2 por separado: no es un capricho

Un ±180° o un espejo son **anatómicamente imposibles** en una VFSS lateral: la columna
cervical tiene orientación fija y el modelo nunca verá un frame invertido. Si solo
corrieras C2 y saliera peor, no podrías distinguir *"más augmentation perjudica"* de
*"augmentation imposible perjudica"*. C1 aísla eso.

## Coste

```
A baseline  3 runs      D bs=1  3 runs      B zscore 3 runs
C1          3 runs      C2      3 runs
                                            = 15 runs, ~4 h
```

El brazo **D va primero**, para que termine temprano si Colab se corta.


In [ ]:
# ============================================================
# SETUP - correr una vez tras cada reinicio del runtime
# No entrena nada.
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

!git clone https://github.com/sebastianquispearias/tesis-seg.git
%cd tesis-seg
!pip install -q -r requirements.txt

import torch, sys
print("Python    :", sys.version)
print("torch     :", torch.__version__)
print("CUDA      :", torch.version.cuda)
!git log --oneline -1
!nvidia-smi | grep -E "NVIDIA|Driver Version|CUDA Version"

import sys
sys.path.append("/content/tesis-seg")

import json, os, time, glob, statistics

from src.defaults import get_default_config, summarize_config
from src.augmentations import get_supervised_train_augmentation
from src.datasets import build_supervised_datasets, build_dataloaders
from src.train import run_training
from src.evaluate import evaluate_checkpoint

# --- GUARDIAN 1: el codigo clonado debe traer los perfiles nuevos ---
# El 2026-08-16 se perdieron 20 runs porque un flag se ignoro en silencio.
import inspect
from src import augmentations as _ag
from src import preprocessing as _pp
assert "get_nnunet_style_augmentation" in inspect.getsource(_ag), (
    "CODIGO VIEJO: src/augmentations.py no tiene los perfiles de nnU-Net. "
    "Borra /content/tesis-seg, vuelve a clonar y reinicia el entorno.")
assert "image_norm" in inspect.getsource(_pp.preprocess_image_and_mask), (
    "CODIGO VIEJO: preprocess_image_and_mask no acepta image_norm.")
print("OK: el codigo clonado trae aug_profile e image_norm.")

# --- GUARDIAN 2: dejar constancia del entorno ---
# Colab cambio de stack entre julio y septiembre de 2026 y albumentations 2.x
# DESCARTA EN SILENCIO argumentos que la 1.x aceptaba (var_limit, value,
# mask_value). Por eso este notebook NO compara contra runs_final_v1, que se
# entreno con albumentations 1.3.1: entrena su propio baseline en esta sesion.
import albumentations as _alb
import segmentation_models_pytorch as _smp
ENTORNO = {"python": sys.version.split()[0], "torch": torch.__version__,
           "albumentations": _alb.__version__, "smp": _smp.__version__,
           "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}
print()
print("ENTORNO DE ESTA SESION")
for k, v in ENTORNO.items():
    print("   %-16s %s" % (k, v))
print()
print("runs_final_v1 se entreno con: albumentations 1.3.1, smp 0.3.3, torch 2.2.1.")
print("Si lo de arriba no coincide, es NORMAL y por eso hay un brazo baseline aqui.")


---
### Paso 1 - Verificar ANTES de gastar GPU

No entrena. Comprueba que los tres perfiles construyen con la albumentations que
Colab acaba de instalar, que el z-score normaliza de verdad, y que el valor por
defecto no se movió.


In [ ]:
# ============================================================
# VERIFICACION - NO ENTRENA. Tarda ~1 minuto.
# ============================================================
import numpy as np
from src.preprocessing import preprocess_image_and_mask

BASE = "/content/drive/MyDrive/UNM_vertebras_seg_v3"

_cfg = get_default_config()
_cfg["img_root"] = BASE
_cfg["msk_root"] = BASE
_cfg["num_workers"] = 0
_cfg["use_semi"] = False

_tr, _, _ = build_supervised_datasets(_cfg, train_tf=None)
_img = np.array(_tr[0]["image"])
print("PUERTA 1: el default no se movio")
print("   min %.6f  max %.6f  media %.6f" % (_img.min(), _img.max(), _img.mean()))
assert _img.min() >= -1e-6 and _img.max() <= 1.0 + 1e-6, "el default cambio"
print("   OK: sigue en [0,1]\n")

_cz = dict(_cfg); _cz["image_norm"] = "zscore"
_tz, _, _ = build_supervised_datasets(_cz, train_tf=None)
_imz = np.array(_tz[0]["image"])
print("PUERTA 2: el z-score normaliza")
print("   media %.6f  std %.6f  min %.3f  max %.3f" % (_imz.mean(), _imz.std(), _imz.min(), _imz.max()))
assert abs(_imz.mean()) < 1e-4 and abs(_imz.std() - 1.0) < 1e-3, "el zscore no normaliza"
print("   OK\n")

print("PUERTA 3: un valor invalido falla en vez de pasar en silencio")
try:
    preprocess_image_and_mask(np.zeros((64, 64, 3), np.uint8),
                              np.zeros((64, 64), np.uint8),
                              target_size=(32, 32), image_norm="disparate")
    raise AssertionError("acepto un valor invalido")
except ValueError as e:
    print("   OK: %s\n" % e)

print("PUERTA 4: los tres perfiles construyen y se aplican")
_im = np.random.randint(0, 255, (320, 320, 3), dtype=np.uint8)
_mk = np.zeros((320, 320), np.uint8); _mk[100:200, 120:180] = 255
for _p in ("base", "nnunet_moderate", "nnunet_full"):
    _c = dict(_cfg); _c["aug_profile"] = _p
    _a = get_supervised_train_augmentation(_c)
    _o = _a(image=_im, mask=_mk)
    print("   %-16s %d transformaciones -> %s" % (_p, len(_a.transforms), [type(t).__name__ for t in _a.transforms]))
assert len(get_supervised_train_augmentation({**_cfg, "aug_profile": "nnunet_full"}).transforms) > \
       len(get_supervised_train_augmentation({**_cfg, "aug_profile": "base"}).transforms), \
       "el perfil nnunet_full no anade nada"
print("   OK\n")
print("TODO VERIFICADO. Se puede entrenar.")


---
### Paso 1b - Los parámetros

Todo lo que se puede tocar está aquí y solo aquí.


In [ ]:
# ============================================================
# PARAMETROS DEL EXPERIMENTO
# ============================================================
BASE     = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
ROTULOS  = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
OUT_ROOT = f"{BASE}/runs_nnunet_ablation"        # directorio NUEVO

SEMILLAS = [0, 1, 2]

# Los brazos, en orden de ejecucion. D va primero a proposito.
# Cada valor es el conjunto de cambios sobre la config del supervisado.
BRAZOS = [
    ("D_bs1",            {"batch_size": 1}),
    ("A_baseline",       {}),
    ("B_zscore",         {"image_norm": "zscore"}),
    ("C1_aug_moderate",  {"aug_profile": "nnunet_moderate"}),
    ("C2_aug_full",      {"aug_profile": "nnunet_full"}),
]

BRAZO_BASELINE = "A_baseline"     # contra el que se comparan los demas

print("brazos   :", [b[0] for b in BRAZOS])
print("semillas :", SEMILLAS)
print("salida   :", OUT_ROOT)
print("runs     : %d  (~%.1f h a 16 min por run)" % (len(BRAZOS) * len(SEMILLAS), len(BRAZOS) * len(SEMILLAS) * 16 / 60))


---
### Paso 1c - El cuerpo de un run

Una sola función. El bloque de `cfg` es el del supervisado de `runs_final_v1/supervised`
copiado tal cual, y `cambios` es lo **único** que distingue un brazo de otro.
Reanudable: si Colab se corta, al reejecutar salta lo que ya terminó.


In [ ]:
def run_arm(nombre, semilla, cambios, verbose=False):
    """Entrena un supervisado UNM aplicando `cambios` sobre la config del baseline.

    `cambios` es el unico sitio donde los brazos se diferencian. Devuelve el exp_dir.
    Si el run ya estaba completo no reentrena nada.
    """
    import gc; gc.collect()
    torch.cuda.empty_cache()

    cfg = get_default_config()
    cfg["img_root"]    = BASE
    cfg["msk_root"]    = BASE
    cfg["rotulos_dir"] = ROTULOS
    cfg["exp_dir"]     = f"{OUT_ROOT}/{nombre}/seed_{semilla}"

    cfg["arch"]      = "unetpp"
    cfg["backbone"]  = "efficientnet-b3"
    cfg["n_classes"] = 1

    cfg["seed"]                 = semilla
    cfg["use_semi"]             = False
    cfg["use_temp_consistency"] = False
    cfg["lambda_u"]             = 0.0
    cfg["lambda_t"]             = 0.0

    cfg["image_preproc"]  = "base"
    cfg["mask_smoothing"] = "none"
    cfg["use_fixed_crop"] = False
    cfg["target_size"]    = (320, 320)
    cfg["use_pad"]        = True
    cfg["imagenet_norm"]  = False

    cfg["batch_size"]     = 5
    cfg["num_workers"]    = 4
    cfg["drop_last"]      = True
    cfg["num_augmented"]  = 5
    cfg["lr"]             = 1e-3
    cfg["weight_decay"]   = 1e-4
    cfg["epochs"]         = 2000
    cfg["warmup_epochs"]  = 10
    cfg["patience_es"]    = 40
    cfg["eval_threshold"] = 0.5

    cfg["save_preds_vis"] = False
    cfg["run_ruler_eval"] = True

    # --- LO UNICO QUE DISTINGUE A ESTE BRAZO ---
    cfg.update(cambios)

    _exp = cfg["exp_dir"]
    _best    = os.path.isfile(os.path.join(_exp, "best_model.pt"))
    _metrics = os.path.isfile(os.path.join(_exp, "test_metrics.csv"))
    _summary = os.path.isfile(os.path.join(_exp, "run_summary.txt"))
    _report  = len(glob.glob(os.path.join(_exp, "*_run_report.json"))) > 0

    if _best and not _summary:
        print(f"AVISO {nombre}/seed_{semilla}: best_model.pt sin run_summary.txt.")
        print("      Run cortado a medias. Se reentrena desde cero.")

    if _best and _metrics and _summary:
        print(f"Skipping {nombre}/seed_{semilla}: run completo detectado")
        return _exp

    if verbose:
        print(summarize_config(cfg))

    # guardian en caliente: los cambios de este brazo tienen que estar en el cfg
    for k, v in cambios.items():
        assert cfg[k] == v, f"FALLO: cfg[{k}] es {cfg[k]!r}, no {v!r}"
    print(f"guardian OK: {nombre} -> {cambios}")

    train_tf = get_supervised_train_augmentation(cfg)
    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    loaders = build_dataloaders(cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds)

    # guardian en caliente 2: el batch que sale del loader debe tener el tamano pedido
    _bx = next(iter(loaders["train_loader"]))["image"]
    assert _bx.shape[0] == cfg["batch_size"], (
        f"FALLO: el loader entrega {_bx.shape[0]} imagenes, no {cfg['batch_size']}.")
    assert tuple(_bx.shape[2:]) == tuple(cfg["target_size"]), "FALLO: resolucion inesperada"

    _t0 = time.time()
    if _best and _summary and (not _metrics or not _report):
        from src.models import create_model
        _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
        results = evaluate_checkpoint(cfg, _m, loaders,
                                      os.path.join(_exp, "best_model.pt"), [])
    else:
        art = run_training(cfg, loaders)
        results = evaluate_checkpoint(cfg, art["model"], loaders,
                                      art["best_path"], art["history"])
    print(f"[{nombre}/seed_{semilla}] {(time.time()-_t0)/60:.1f} min")
    print(results)
    return _exp


print("run_arm definida.")


---
## Paso 2 - Los 15 runs (~4 h)

`D_bs1` va primero. Si Colab se cae de madrugada, ese al menos estará hecho, y es el
que decide si la resolución sigue siendo una cuestión abierta.


In [ ]:
# === PASO 2: todos los brazos ===
_fallos = []
for _nombre, _cambios in BRAZOS:
    for _seed in SEMILLAS:
        print("=" * 70)
        print(f">>> {_nombre}  seed={_seed}  cambios={_cambios}")
        print("=" * 70)
        try:
            run_arm(_nombre, _seed, _cambios)
        except Exception as e:
            print(f"FALLO en {_nombre}/seed_{_seed}: {type(e).__name__}: {e}")
            _fallos.append((_nombre, _seed, repr(e)))

print()
print("terminado. fallos:", len(_fallos))
for f in _fallos:
    print("  ", f)


---
## Paso 3 - El resumen

Compara cada brazo contra `A_baseline`, **entrenado en esta misma sesión**. La celda
comprueba además que todos los runs comparados salieron del mismo entorno, y avisa si
no. También imprime, solo como referencia y marcado como tal, el baseline de julio.


In [ ]:
# === PASO 3: RESUMEN (solo lectura) ===
def leer_report(exp_dir):
    _f = glob.glob(os.path.join(exp_dir, "*_run_report.json"))
    if not _f:
        return None
    with open(_f[0], encoding="utf-8") as fh:
        return json.load(fh)


def entorno_de(exp_dir):
    _f = os.path.join(exp_dir, "debug_fingerprint.json")
    if not os.path.isfile(_f):
        return None
    with open(_f, encoding="utf-8") as fh:
        e = json.load(fh).get("pre_run", {}).get("environment", {})
    return (e.get("albumentations"), e.get("segmentation_models_pytorch"), e.get("torch"))


def metricas(exp_dir):
    r = leer_report(exp_dir)
    if r is None:
        return None
    t = r.get("test_metrics") or {}
    b = r.get("test_boundary_metrics") or {}
    return {"f1": t.get("sample_mean_f1"), "assd": b.get("assd_mean_px"),
            "hd95": b.get("hd95_mean_px")}


def agregar(nombre):
    dirs = [f"{OUT_ROOT}/{nombre}/seed_{s}" for s in SEMILLAS]
    vals = [metricas(d) for d in dirs]
    vals = [v for v in vals if v and v["f1"] is not None]
    envs = {entorno_de(d) for d in dirs if entorno_de(d)}
    if not vals:
        return None, envs
    out = {"n": len(vals)}
    for k in ("f1", "assd", "hd95"):
        xs = [v[k] for v in vals if v[k] is not None]
        out[k] = (statistics.mean(xs), statistics.stdev(xs) if len(xs) > 1 else 0.0) if xs else (None, None)
    return out, envs


_todos_env = set()
_res = {}
for _nombre, _ in BRAZOS:
    _ag2, _ev = agregar(_nombre)
    _res[_nombre] = _ag2
    _todos_env |= _ev

print("ENTORNOS ENCONTRADOS (albumentations, smp, torch):")
for e in sorted(_todos_env, key=str):
    print("   ", e)
if len(_todos_env) > 1:
    print()
    print("   *** AVISO: los brazos NO comparten entorno. La comparacion mezcla el")
    print("       cambio con una actualizacion de librerias. Reentrena los brazos")
    print("       afectados en una sola sesion antes de concluir nada.")
print()

_base = _res.get(BRAZO_BASELINE)


def fmt(t):
    return "%.4f +/- %.4f" % t if t and t[0] is not None else "-"


print("%-20s %3s %18s %18s %18s" % ("brazo", "n", "F1", "ASSD (px)", "HD95 (px)"))
print("-" * 82)
for _nombre, _ in BRAZOS:
    a = _res[_nombre]
    if a is None:
        print("%-20s %3s %18s" % (_nombre, "-", "sin runs"))
        continue
    print("%-20s %3d %18s %18s %18s" % (_nombre, a["n"], fmt(a["f1"]), fmt(a["assd"]), fmt(a["hd95"])))
    if _base and _nombre != BRAZO_BASELINE:
        for _k, _n2 in (("f1", "F1"), ("assd", "ASSD"), ("hd95", "HD95")):
            if a[_k][0] is not None and _base[_k][0] is not None:
                _d = a[_k][0] - _base[_k][0]
                _mejor = (_d > 0) if _k == "f1" else (_d < 0)
                print("       %-5s delta = %+8.4f  (%s)" % (_n2, _d, "mejor" if _mejor else "peor"))
print("-" * 82)
print()
print("REFERENCIA, NO COMPARABLE (otro entorno, julio 2026):")
_ref = [metricas(d) for d in sorted(glob.glob(f"{BASE}/runs_final_v1/supervised/seed_*"))]
_ref = [v["f1"] for v in _ref if v and v["f1"] is not None]
if _ref:
    print("   runs_final_v1/supervised  n=%d  F1 = %.4f +/- %.4f" % (len(_ref), statistics.mean(_ref), statistics.stdev(_ref)))
print()
print("COMO LEERLO")
print("  D_bs1 cerca de .75  -> el batch explica el resultado de 1024 y la RESOLUCION")
print("                         NUNCA SE PROBO. Siguiente paso: 512 con bs=2, o 1024")
print("                         con GroupNorm en vez de BatchNorm.")
print("  D_bs1 cerca de .80  -> entonces si habia algo en la resolucion.")
print("  C1 mejor y C2 peor  -> mas augmentation ayuda, pero el espejo y el +/-180")
print("                         son geometria imposible en una VFSS lateral.")
print("  Los resultados van al banco de preguntas de defensa. NO se meten en la tesis")
print("  sin decidirlo aparte.")
